# Chương 7: Xây dựng Ứng dụng Trò chuyện
## Bắt đầu nhanh với API OpenAI

Notebook này được điều chỉnh từ [Kho mẫu Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) bao gồm các notebook truy cập các dịch vụ [Azure OpenAI](aoai-assignment.ipynb).

API OpenAI Python cũng hoạt động với các mô hình Azure OpenAI, với một vài chỉnh sửa nhỏ. Tìm hiểu thêm về những khác biệt tại đây: [Cách chuyển đổi giữa các điểm cuối OpenAI và Azure OpenAI với Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Tổng quan  
"Mô hình ngôn ngữ lớn là các hàm ánh xạ từ văn bản sang văn bản. Cho một chuỗi văn bản đầu vào, một mô hình ngôn ngữ lớn cố gắng dự đoán văn bản sẽ xuất hiện tiếp theo"(1). Bài ghi chú "khởi động nhanh" này sẽ giới thiệu cho người dùng các khái niệm cấp cao về LLM, các yêu cầu gói cốt lõi để bắt đầu với AML, một giới thiệu nhẹ về thiết kế prompt, và một số ví dụ ngắn về các trường hợp sử dụng khác nhau. 


## Mục Lục  

[Tổng quan](#overview)  
[Cách sử dụng Dịch vụ OpenAI](#how-to-use-openai-service)  
[1. Tạo dịch vụ OpenAI của bạn](#1.-creating-your-openai-service)  
[2. Cài đặt](#2.-installation)    
[3. Thông tin xác thực](#3.-credentials)  

[Các trường hợp sử dụng](#use-cases)    
[1. Tóm tắt văn bản](#1.-summarize-text)  
[2. Phân loại văn bản](#2.-classify-text)  
[3. Tạo tên sản phẩm mới](#3.-generate-new-product-names)  
[4. Tinh chỉnh bộ phân loại](#4.fine-tune-a-classifier)  

[Tài liệu tham khảo](#references)


### Xây dựng prompt đầu tiên của bạn  
Bài tập ngắn này sẽ cung cấp một giới thiệu cơ bản về cách gửi prompt đến một mô hình OpenAI cho một nhiệm vụ đơn giản "tóm tắt".  


**Các bước**:  
1. Cài đặt thư viện OpenAI trong môi trường python của bạn  
2. Tải các thư viện hỗ trợ tiêu chuẩn và đặt các thông tin bảo mật OpenAI điển hình cho Dịch vụ OpenAI mà bạn đã tạo  
3. Chọn một mô hình cho nhiệm vụ của bạn  
4. Tạo một prompt đơn giản cho mô hình  
5. Gửi yêu cầu của bạn tới API của mô hình!  


### 1. Cài đặt OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Nhập các thư viện trợ giúp và khởi tạo thông tin xác thực


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Tìm mô hình phù hợp  
Các mô hình như GPT-4o và GPT-4o mini có thể hiểu và tạo ra ngôn ngữ tự nhiên, và có sẵn trên nền tảng OpenAI với các cấp độ sức mạnh và tốc độ khác nhau phù hợp cho các nhiệm vụ khác nhau.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Thiết kế Prompt  

"Điều kỳ diệu của các mô hình ngôn ngữ lớn là bằng cách được huấn luyện để giảm thiểu lỗi dự đoán này trên một lượng lớn văn bản, các mô hình cuối cùng học được những khái niệm hữu ích cho các dự đoán đó. Ví dụ, chúng học được các khái niệm như"(1):

* cách đánh vần
* cách hoạt động của ngữ pháp
* cách diễn đạt lại
* cách trả lời câu hỏi
* cách giữ cuộc trò chuyện
* cách viết bằng nhiều ngôn ngữ
* cách lập trình
* v.v.

#### Cách kiểm soát một mô hình ngôn ngữ lớn  
"Trong tất cả các đầu vào cho một mô hình ngôn ngữ lớn, yếu tố ảnh hưởng nhất chính là prompt văn bản(1).

Các mô hình ngôn ngữ lớn có thể được gợi câu để tạo đầu ra theo một vài cách:

Hướng dẫn: Nói cho mô hình biết bạn muốn gì
Hoàn thành: Gợi mô hình hoàn thành phần đầu của những gì bạn muốn
Minh họa: Cho mô hình thấy bạn muốn gì, với:
Một vài ví dụ trong prompt
Hàng trăm hoặc hàng nghìn ví dụ trong bộ dữ liệu huấn luyện tinh chỉnh"



#### Có ba nguyên tắc cơ bản để tạo prompt:

**Hiển thị và nói rõ**. Làm rõ bạn muốn gì bằng cách dùng hướng dẫn, ví dụ, hoặc kết hợp cả hai. Nếu bạn muốn mô hình xếp hạng một danh sách theo thứ tự chữ cái hoặc phân loại một đoạn văn theo cảm xúc, hãy cho nó thấy đó là điều bạn muốn.

**Cung cấp dữ liệu chất lượng**. Nếu bạn đang cố xây dựng một bộ phân loại hoặc để mô hình theo một mẫu nhất định, hãy chắc chắn có đủ ví dụ. Hãy chắc rằng bạn đã đọc kỹ các ví dụ — mô hình thường đủ thông minh để nhận ra các lỗi chính tả cơ bản và vẫn trả lời bạn, nhưng nó cũng có thể nghĩ điều đó là có chủ ý và điều này có thể ảnh hưởng đến câu trả lời.

**Kiểm tra cài đặt của bạn.** Cài đặt nhiệt độ và top_p điều khiển mức độ xác định của mô hình khi tạo phản hồi. Nếu bạn yêu cầu một phản hồi mà chỉ có một câu trả lời đúng duy nhất, thì bạn nên đặt các giá trị này thấp hơn. Nếu bạn muốn có các phản hồi đa dạng hơn, thì bạn có thể đặt chúng cao hơn. Sai lầm lớn nhất mà mọi người thường mắc phải với các cài đặt này là cho rằng chúng là các điều khiển về "sự thông minh" hoặc "sự sáng tạo".


Source: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Gửi!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Gọi lại cùng một hàm, kết quả có giống nhau không?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Tóm tắt văn bản  
#### Thử thách  
Tóm tắt văn bản bằng cách thêm 'tl;dr:' vào cuối một đoạn văn bản. Hãy chú ý cách mô hình hiểu và thực hiện nhiều nhiệm vụ mà không cần hướng dẫn thêm. Bạn có thể thử nghiệm với các lời nhắc mô tả hơn tl;dr để thay đổi hành vi của mô hình và tùy chỉnh bản tóm tắt bạn nhận được(3).  

Công trình gần đây đã chứng minh sự cải thiện đáng kể trên nhiều nhiệm vụ và chuẩn đánh giá NLP bằng cách huấn luyện sơ bộ trên một tập văn bản lớn, sau đó tinh chỉnh trên một nhiệm vụ cụ thể. Mặc dù kiến trúc thường không phụ thuộc vào nhiệm vụ, phương pháp này vẫn đòi hỏi các bộ dữ liệu tinh chỉnh riêng cho từng nhiệm vụ với hàng nghìn hoặc hàng chục nghìn ví dụ. Ngược lại, con người thường có thể thực hiện một nhiệm vụ ngôn ngữ mới chỉ từ vài ví dụ hoặc từ các hướng dẫn đơn giản - điều mà các hệ thống NLP hiện tại vẫn đang gặp khó khăn. Ở đây, chúng tôi cho thấy việc mở rộng kích thước mô hình ngôn ngữ cải thiện rất nhiều hiệu suất thực hiện nhiệm vụ ít ví dụ không phụ thuộc nhiệm vụ, đôi khi còn đạt được hiệu quả cạnh tranh với các phương pháp tinh chỉnh tiên tiến trước đó. 



Tl;dr


# Bài tập cho nhiều trường hợp sử dụng  
1. Tóm tắt văn bản  
2. Phân loại văn bản  
3. Tạo tên sản phẩm mới


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Phân loại văn bản  
#### Thử thách  
Phân loại các mục vào các danh mục được cung cấp khi suy luận. Trong ví dụ dưới đây, chúng tôi cung cấp cả các danh mục và văn bản để phân loại trong lời nhắc (*playground_reference). 

Yêu cầu của khách hàng: Xin chào, một trong những phím trên bàn phím laptop của tôi gần đây bị hỏng và tôi cần một bộ thay thế:

Danh mục phân loại:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Tạo Tên Sản Phẩm Mới
#### Thách thức
Tạo tên sản phẩm từ các từ ví dụ. Ở đây, chúng tôi bao gồm trong lời nhắc thông tin về sản phẩm mà chúng tôi sẽ tạo tên cho nó. Chúng tôi cũng cung cấp một ví dụ tương tự để cho thấy mẫu mà chúng tôi muốn nhận được. Chúng tôi cũng đã đặt giá trị nhiệt độ cao để tăng tính ngẫu nhiên và các phản hồi sáng tạo hơn.

Mô tả sản phẩm: Máy làm sữa lắc tại nhà
Từ khóa gốc: nhanh, lành mạnh, gọn nhẹ.
Tên sản phẩm: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Mô tả sản phẩm: Một đôi giày có thể phù hợp với mọi kích cỡ chân.
Từ khóa gốc: thích nghi, vừa vặn, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Tài liệu tham khảo  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Cổng thông tin Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Các phương pháp tốt nhất để tinh chỉnh các mô hình GPT nhằm phân loại văn bản](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Để được hỗ trợ thêm  
[Nhóm Thương mại hóa OpenAI](AzureOpenAITeam@microsoft.com) 


# Những người đóng góp
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
